# 06 Lapse Risk Model

This notebook builds a rolling-snapshot ML model to predict whether an existing customer is likely to lapse, defined as no valid purchase in the next 90 days.

Inputs:

- `../data/processed/clean_transactions.parquet`

Outputs:

- `../data/processed/lapse_risk_model_metrics.csv`
- `../data/processed/lapse_risk_scores.csv`
- `../data/processed/lapse_risk_lift.csv`
- `../data/processed/lapse_risk_feature_importance.csv`


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, brier_score_loss, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROCESSED_DIR = Path(os.environ.get("PROCESSED_DIR", "../data/processed"))
TRANSACTIONS_PATH = PROCESSED_DIR / "clean_transactions.parquet"
RANDOM_STATE = 42
TARGET_WINDOW_DAYS = 90

if not TRANSACTIONS_PATH.exists():
    raise FileNotFoundError(f"Required input not found: {TRANSACTIONS_PATH}. Run notebook 01 first.")

## Build Customer Orders

The model starts from valid customer order lines and groups them to invoice-level orders. The rolling snapshots use only orders available before each snapshot date.

In [ ]:
transactions = pd.read_parquet(TRANSACTIONS_PATH)
transactions.columns = transactions.columns.str.strip().str.lower().str.replace(" ", "_")
transactions = transactions.rename(columns={"invoiceno": "invoice_no", "stockcode": "stock_code", "invoicedate": "invoice_date", "customerid": "customer_id"})
transactions["invoice_date"] = pd.to_datetime(transactions["invoice_date"])
valid_lines = transactions[transactions["is_valid_order_line"]].copy()

orders = (
    valid_lines.groupby(["customer_id", "invoice_no"], as_index=False)
    .agg(order_date=("invoice_date", "min"), order_revenue=("analytical_revenue", "sum"), order_quantity=("quantity", "sum"), order_lines=("stock_code", "size"), country=("country", lambda values: values.mode().iloc[0] if not values.mode().empty else values.iloc[0]))
)
orders = orders[orders["order_revenue"] > 0].copy()
orders = orders.sort_values(["customer_id", "order_date", "invoice_no"]).reset_index(drop=True)
orders["order_rank"] = orders.groupby("customer_id").cumcount() + 1

print(f"Orders: {len(orders):,}")
print(f"Customers: {orders['customer_id'].nunique():,}")
orders.head()

## Rolling Snapshot Dataset

Each monthly snapshot represents customers known before that date. The target is `lapsed_next_90_days`, equal to 1 when the customer has no valid purchase in the next 90 days. Snapshots without a full 90-day observation window are excluded.

In [ ]:
min_date = orders["order_date"].min().normalize()
max_date = orders["order_date"].max().normalize()
snapshot_dates = pd.date_range(min_date + pd.DateOffset(months=3), max_date - pd.Timedelta(days=TARGET_WINDOW_DAYS), freq="MS")

rows = []
for snapshot_date in snapshot_dates:
    history = orders[orders["order_date"] < snapshot_date].copy()
    future = orders[(orders["order_date"] >= snapshot_date) & (orders["order_date"] <= snapshot_date + pd.Timedelta(days=TARGET_WINDOW_DAYS))]
    if history.empty:
        continue

    customer_features = (
        history.groupby("customer_id", as_index=False)
        .agg(total_orders=("invoice_no", "nunique"), revenue=("order_revenue", "sum"), first_purchase=("order_date", "min"), last_purchase=("order_date", "max"), avg_order_value=("order_revenue", "mean"), avg_order_quantity=("order_quantity", "mean"), avg_order_lines=("order_lines", "mean"), country=("country", lambda values: values.mode().iloc[0] if not values.mode().empty else values.iloc[0]))
    )
    customer_features["snapshot_date"] = snapshot_date
    customer_features["recency_days"] = (snapshot_date - customer_features["last_purchase"]).dt.days
    customer_features["customer_age_days"] = (snapshot_date - customer_features["first_purchase"]).dt.days
    customer_features["active_days"] = (customer_features["last_purchase"] - customer_features["first_purchase"]).dt.days
    customer_features["avg_days_between_orders"] = np.where(customer_features["total_orders"] > 1, customer_features["active_days"] / (customer_features["total_orders"] - 1), np.nan)
    customer_features["orders_per_active_day"] = customer_features["total_orders"] / customer_features["customer_age_days"].clip(lower=1)
    customer_features["is_repeat_customer"] = customer_features["total_orders"].gt(1).astype(int)

    second_orders = history[history["order_rank"] == 2][["customer_id", "order_date"]].rename(columns={"order_date": "second_order_date"})
    first_orders = history[history["order_rank"] == 1][["customer_id", "order_date"]].rename(columns={"order_date": "first_order_date"})
    days_to_second = second_orders.merge(first_orders, on="customer_id", how="left")
    days_to_second["days_to_second_purchase"] = (days_to_second["second_order_date"] - days_to_second["first_order_date"]).dt.days
    customer_features = customer_features.merge(days_to_second[["customer_id", "days_to_second_purchase"]], on="customer_id", how="left")

    customer_features = customer_features.sort_values("revenue", ascending=False).reset_index(drop=True)
    customer_features["revenue_rank_pct"] = (customer_features.index + 1) / len(customer_features)
    customer_features["is_high_value"] = customer_features["revenue_rank_pct"].le(0.10).astype(int)

    future_customers = set(future["customer_id"].unique())
    customer_features["lapsed_next_90_days"] = (~customer_features["customer_id"].isin(future_customers)).astype(int)
    rows.append(customer_features)

snapshot_data = pd.concat(rows, ignore_index=True)
snapshot_data["log_revenue"] = np.log1p(snapshot_data["revenue"])
snapshot_data["log_avg_order_value"] = np.log1p(snapshot_data["avg_order_value"])
snapshot_data["snapshot_month"] = snapshot_data["snapshot_date"].dt.month.astype(str)
snapshot_data["country_group"] = np.where(snapshot_data["country"].eq("United Kingdom"), "United Kingdom", "International")

print(f"Snapshot rows: {len(snapshot_data):,}")
print(f"Snapshots: {snapshot_data['snapshot_date'].nunique():,}")
print(f"Lapse target rate: {snapshot_data['lapsed_next_90_days'].mean():.2%}")
snapshot_data.head()

## Train/Test Split

The model is trained on earlier monthly snapshots and tested on later snapshots to mimic a real lifecycle scoring workflow.

In [ ]:
feature_columns = ["recency_days", "customer_age_days", "active_days", "avg_days_between_orders", "orders_per_active_day", "total_orders", "log_revenue", "log_avg_order_value", "avg_order_quantity", "avg_order_lines", "days_to_second_purchase", "is_repeat_customer", "is_high_value", "country_group", "snapshot_month"]
target_column = "lapsed_next_90_days"

unique_snapshots = sorted(snapshot_data["snapshot_date"].unique())
split_snapshot = unique_snapshots[int(len(unique_snapshots) * 0.70)]
train_data = snapshot_data[snapshot_data["snapshot_date"] < split_snapshot].copy()
test_data = snapshot_data[snapshot_data["snapshot_date"] >= split_snapshot].copy()

X_train = train_data[feature_columns]
y_train = train_data[target_column]
X_test = test_data[feature_columns]
y_test = test_data[target_column]

print(f"Split snapshot: {pd.Timestamp(split_snapshot).date()}")
print(f"Train rows: {len(train_data):,}, target rate: {y_train.mean():.2%}")
print(f"Test rows: {len(test_data):,}, target rate: {y_test.mean():.2%}")

## Train Models

The selected model prioritises top-20% precision because CRM teams usually act on the highest-risk customer bands rather than the entire scored population.

In [ ]:
numeric_features = [col for col in feature_columns if col not in ["country_group", "snapshot_month"]]
categorical_features = ["country_group", "snapshot_month"]

numeric_preprocess = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_preprocess = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
preprocess = ColumnTransformer(transformers=[("numeric", numeric_preprocess, numeric_features), ("categorical", categorical_preprocess, categorical_features)])

models = {
    "logistic_regression": Pipeline(steps=[("preprocess", preprocess), ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))]),
    "random_forest": Pipeline(steps=[("preprocess", preprocess), ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=50, class_weight="balanced", random_state=RANDOM_STATE))]),
    "hist_gradient_boosting": Pipeline(steps=[("preprocess", preprocess), ("model", HistGradientBoostingClassifier(learning_rate=0.05, max_iter=200, min_samples_leaf=50, l2_regularization=0.1, random_state=RANDOM_STATE))]),
}

def precision_at_fraction(y_true, probabilities, fraction):
    cutoff = max(1, int(np.ceil(len(y_true) * fraction)))
    top_index = np.argsort(probabilities)[::-1][:cutoff]
    return y_true.iloc[top_index].mean()

model_results = []
fitted_models = {}
for model_name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[model_name] = model
    test_probability = model.predict_proba(X_test)[:, 1]
    test_prediction = (test_probability >= 0.5).astype(int)
    model_results.append({
        "model": model_name,
        "train_rows": len(train_data),
        "test_rows": len(test_data),
        "test_target_rate": y_test.mean(),
        "roc_auc": roc_auc_score(y_test, test_probability),
        "pr_auc": average_precision_score(y_test, test_probability),
        "brier_score": brier_score_loss(y_test, test_probability),
        "accuracy_at_50pct_threshold": accuracy_score(y_test, test_prediction),
        "precision_at_50pct_threshold": precision_score(y_test, test_prediction, zero_division=0),
        "recall_at_50pct_threshold": recall_score(y_test, test_prediction, zero_division=0),
        "precision_top_10pct": precision_at_fraction(y_test, test_probability, 0.10),
        "precision_top_20pct": precision_at_fraction(y_test, test_probability, 0.20),
        "precision_top_30pct": precision_at_fraction(y_test, test_probability, 0.30),
    })

model_metrics = pd.DataFrame(model_results).sort_values(["precision_top_20pct", "roc_auc"], ascending=False).reset_index(drop=True)
selected_model_name = model_metrics.loc[0, "model"]
selected_model = fitted_models[selected_model_name]
model_metrics.to_csv(PROCESSED_DIR / "lapse_risk_model_metrics.csv", index=False)

print(f"Selected model: {selected_model_name}")
model_metrics

## Score Customers and Build Lift

The latest available full-observation snapshot is used as the customer scoring output. Higher scores represent higher predicted lapse risk.

In [ ]:
snapshot_data["lapse_risk_score"] = selected_model.predict_proba(snapshot_data[feature_columns])[:, 1]
snapshot_data["data_split"] = np.where(snapshot_data["snapshot_date"] < split_snapshot, "train", "test")

test_scored = snapshot_data[snapshot_data["data_split"] == "test"].copy()
test_scored = test_scored.sort_values("lapse_risk_score", ascending=False).reset_index(drop=True)
test_scored["risk_decile"] = pd.qcut(test_scored.index + 1, 10, labels=["Top 10%", "10-20%", "20-30%", "30-40%", "40-50%", "50-60%", "60-70%", "70-80%", "80-90%", "Bottom 10%"])
overall_test_lapse_rate = test_scored[target_column].mean()
lift = (test_scored.groupby("risk_decile", observed=False).agg(customers=("customer_id", "count"), observed_lapse_rate=(target_column, "mean"), avg_risk_score=("lapse_risk_score", "mean"), high_value_customers=("is_high_value", "sum")).reset_index())
lift["lift_vs_test_average"] = lift["observed_lapse_rate"] / overall_test_lapse_rate
lift.to_csv(PROCESSED_DIR / "lapse_risk_lift.csv", index=False)

latest_snapshot = snapshot_data["snapshot_date"].max()
scores = snapshot_data[snapshot_data["snapshot_date"] == latest_snapshot].copy()
scores["risk_band"] = pd.cut(scores["lapse_risk_score"], bins=[-np.inf, 0.40, 0.65, np.inf], labels=["Low risk", "Medium risk", "High risk"])
scores["recommended_action"] = np.select([scores["is_high_value"].eq(1) & scores["risk_band"].eq("High risk"), scores["risk_band"].eq("High risk"), scores["risk_band"].eq("Medium risk")], ["Priority high-value retention intervention", "Automated reactivation or replenishment trigger", "Light-touch reminder or nurture"], default="Maintain standard lifecycle messaging")
score_columns = ["customer_id", "snapshot_date", "lapse_risk_score", "risk_band", "recommended_action", "total_orders", "revenue", "recency_days", "customer_age_days", "avg_days_between_orders", "is_repeat_customer", "is_high_value", "country_group", "lapsed_next_90_days"]
scores[score_columns].to_csv(PROCESSED_DIR / "lapse_risk_scores.csv", index=False)

lift

In [ ]:
permutation = permutation_importance(selected_model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, scoring="roc_auc")
feature_importance = pd.DataFrame({"feature": feature_columns, "importance": permutation.importances_mean}).sort_values("importance", ascending=False)
feature_importance.to_csv(PROCESSED_DIR / "lapse_risk_feature_importance.csv", index=False)
feature_importance.head(20)

In [ ]:
expected_outputs = {
    "lapse_risk_model_metrics.csv": {"model", "roc_auc", "precision_top_20pct"},
    "lapse_risk_scores.csv": {"customer_id", "snapshot_date", "lapse_risk_score", "risk_band", "recommended_action"},
    "lapse_risk_lift.csv": {"risk_decile", "observed_lapse_rate", "lift_vs_test_average"},
    "lapse_risk_feature_importance.csv": {"feature", "importance"},
}
for file_name, expected_columns in expected_outputs.items():
    path = PROCESSED_DIR / file_name
    if not path.exists():
        raise FileNotFoundError(f"Expected output was not created: {path}")
    output = pd.read_csv(path)
    missing_columns = sorted(expected_columns - set(output.columns))
    if missing_columns:
        raise ValueError(f"{file_name} is missing columns: {missing_columns}")
    print(f"Validated {file_name}: {output.shape}")